In [ ]:
### Kenyan Parliamentary Hansard Analysis Script

## Nicolas de Zamaroczy

## The purpose of this Python script is to perform simple quantitative text analysis of the parsed Kenyan parliamentary hansards.
# In particular, we track sentiment analysis in a variety of ways:
# - over time;
# - by speaker; and
# - by political party.

# The current goal is to see if parliamentary speech is a leading or lagging indicator of political violence in Kenya.

In [ ]:
### PRELIMINARIES

# --- Dependency Checks ---
# This block checks if several required libraries are installed.
# If not, it uses Python's package manager (pip) to install them.
# This makes the script more portable and easier for others to run.
try:
    import numpy as np # NumPy, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("NumPy not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
    import numpy as np
    print("Installation complete.")

try:
    import pandas as pd # Pandas, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("Pandas not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
    print("Installation complete.")

try:
    from pathlib import Path # PathLib, used to read/write files to a given folder
except ImportError:
    import subprocess
    import sys
    print("PathLib not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pathlib[Path]"])
    from pathlib import Path
    print("Installation complete.")

try:
    from tqdm import tqdm # tqdm, used to create a progress bar
except ImportError:
    import subprocess
    import sys
    print("tqdm not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm[tqdm]"])
    from tqdm import tqdm
    print("Installation complete.")

# Enable tqdm for Pandas
tqdm.pandas()

# Disable Future Deprecation Warnings from getting logged
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
# Load the parsed speeches from the pipeline's database (built by `hansard parse`)

import json
import sqlite3
from contextlib import closing

# --- Settings ---
CHAMBER = "senate"  # "senate", "national_assembly" or "joint"
# Number of speeches to analyse, picked at random. None analyses every speech, which
# takes many hours on a CPU; a few thousand is enough for a quick test run.
SAMPLE_SIZE = None

# Current notebook directory
notebook_dir = Path.cwd()
# Path to the database
db_path = notebook_dir.parent / "outputs" / "hansard.db"
# Check existence and raise an error if missing
if not db_path.exists():
    raise FileNotFoundError(f"Database not found: {db_path}. Run `hansard parse` first.")

# One row per speech, with its document and (when matched) member details joined in
query = """
SELECT s.id AS speech_id, d.filename, d.date, d.chamber, d.sitting,
       s.speaker_raw AS speaker, m.full_name AS matched_name, m.party, m.county,
       m.constituency, s.language_distribution, s.primary_language, s.text AS content
FROM speeches s
JOIN documents d ON d.id = s.document_id
LEFT JOIN members m ON m.id = s.member_id
WHERE d.doc_type = 'hansard' AND d.chamber = ?
ORDER BY d.date, d.id, s.seq
"""
with closing(sqlite3.connect(db_path)) as conn:
    df_blocks = pd.read_sql_query(query, conn, params=[CHAMBER])

if SAMPLE_SIZE:
    df_blocks = (df_blocks.sample(n=min(SAMPLE_SIZE, len(df_blocks)), random_state=42)
                 .sort_values(["date", "speech_id"], ignore_index=True))

# Paragraphs are separated by newlines; the models work on plain running text
df_blocks["content"] = df_blocks["content"].str.replace(r"[\r\n]+", " ", regex=True)

# Language mix as a dict, e.g. {"en": 0.8, "sw": 0.2}
df_blocks["language_distribution"] = df_blocks["language_distribution"].map(json.loads)

# Ensure date column is a datetime
df_blocks["date"] = pd.to_datetime(df_blocks["date"])

df_blocks

In [ ]:
# ==============================================================================
# --- CORRELATING TO POLITICAL VIOLENCE ---
# ==============================================================================

### Get ADM-1 Boundaries for Kenya

import geopandas as gpd
import matplotlib.pyplot as plt

# Path to the shapefile (only need to reference the .shp file)
# These GADM boundaries can't be redistributed, so they aren't in git; see the README.
shapefile_path = notebook_dir.parent / "inputs" / "other_data" / "kenya_adm1_shapefiles" / "KEN_adm1.shp"
# Check existence and raise an error if missing
if not shapefile_path.exists():
    raise FileNotFoundError(f"Shapefile not found: {shapefile_path}")

# Read the shapefile into a GeoDataFrame
gdf = gpd.read_file(shapefile_path)

# Display basic info
print(gdf.head())
print(gdf.crs)

# Simple plot
gdf.plot(edgecolor="black", figsize=(8, 8))
plt.title("Kenya ADM1 Boundaries")
plt.show()

In [ ]:
### Get ACLED Data for Kenya


## There's two ways to do this.
## To get the most up-to-date data, we can hit ACLED's API and pull all Kenyan political violence up to ## days ago (a built-in lag by ACLED).
## Alternatively, if we want to go faster or the API pull isn't working for some reason, we just use the last stored CSV.

# --- Settings ---
# Set to True to download the latest events from ACLED's API. This needs ACLED_USERNAME
# and ACLED_PASSWORD in the project's .env file (see .env.example).
USE_ACLED_API = False
ACLED_START_DATE = "2013-01-01"

# Downloaded ACLED data is saved here. ACLED's terms don't allow redistributing it,
# so this folder's data files are gitignored.
acled_dir = notebook_dir.parent / "inputs" / "other_data"

# Connecting to ACLED's API (https://acleddata.com/api-documentation/getting-started)

import os
from datetime import date

import requests

# Function to get access token using username and password
def get_access_token(username, password, token_url):
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
    }
    data = {
          'username': username,
          'password': password,
          'grant_type': "password",
          'client_id': "acled",
          'scope': "authenticated"
    }

    response = requests.post(token_url, headers=headers, data=data, timeout=60)

    if response.status_code == 200:
        token_data = response.json()
        return token_data['access_token']
    else:
        raise Exception(f"Failed to get access token: {response.status_code} {response.text}")


if USE_ACLED_API:
    # Credentials are read from environment variables (or a local, gitignored .env file).
    try:
        from dotenv import load_dotenv
        load_dotenv(notebook_dir.parent / ".env")
    except ImportError:
        pass

    # Get an access token
    my_token = get_access_token(
        username=os.environ["ACLED_USERNAME"],
        password=os.environ["ACLED_PASSWORD"],
        token_url="https://acleddata.com/oauth/token",
    )

    # The API returns at most `limit` events per request, so request page after page
    limit = 5000
    events = []
    page = 1
    while True:
        response = requests.get(
            "https://acleddata.com/api/acled/read",
            params={
                "_format": "json",
                "country": "Kenya",
                "event_date": f"{ACLED_START_DATE}|{date.today().isoformat()}",
                "event_date_where": "BETWEEN",
                "limit": limit,
                "page": page,
            },
            headers={"Authorization": f"Bearer {my_token}"},
            timeout=120,
        )
        response.raise_for_status()
        payload = response.json()
        if payload.get("status") != 200:
            raise Exception(f"ACLED request failed: {payload}")
        events.extend(payload["data"])
        print(f"Page {page}: {len(payload['data'])} events")
        if len(payload["data"]) < limit:
            break
        page += 1

    # Save the download so later runs can skip the API
    acled_path = acled_dir / f"ACLED Kenya Data through {date.today():%m-%d-%Y}.csv"
    pd.DataFrame(events).to_csv(acled_path, index=False)
    print(f"Saved {len(events)} events to {acled_path}")
else:
    print("Using the most recent stored ACLED CSV (set USE_ACLED_API = True to download new data).")

In [ ]:
## Load raw data from CSV

# The most recent ACLED download in inputs/other_data/ (named "ACLED Kenya Data through <date>.csv")
acled_files = sorted(acled_dir.glob("ACLED Kenya*.csv"), key=lambda p: p.stat().st_mtime)
if not acled_files:
    raise FileNotFoundError(f"No ACLED CSV found in {acled_dir}. Download one from acleddata.com "
                            "(see the README) or set USE_ACLED_API = True in the cell above.")
print(f"Loading {acled_files[-1].name}")
df_incidents = pd.read_csv(acled_files[-1])

# Clean data
df_incidents.columns = df_incidents.columns.str.lower()

# Drop strategic developments
#df_incidents.event_type.unique()
df_incidents = df_incidents[df_incidents['event_type']!='Strategic developments'].copy()

df_incidents

In [ ]:
## Plotting a country's events (with ADM-1 boundaries)

## Note: If a country doesn't have a given type of event in the period in question, you need to hash it out.

## Create an empty figure of the desired size
fig, ax = plt.subplots(figsize=(10,10))

## Specify some parameters
title = 'Instability Events in Kenya (by type)'

## Creating the events plots
# Get the data
df_kenya = df_incidents[['event_type','latitude','longitude','admin1','admin2','admin3','location']].copy()

# Convert df_country_events to a Geopandas df
df_geo = gpd.GeoDataFrame(
    df_kenya
    ,geometry=gpd.points_from_xy(df_kenya.longitude, df_kenya.latitude)
    ,crs="epsg:4326" # We specify that we want to use the same CRS as that which the basemap comes in
)
#df_geo

# Create a plot for Protest events
df_geo[df_geo['event_type'] == 'Protests'].plot(ax=ax, 
                                       markersize=12, 
                                       color='green', 
                                       marker='o', 
                                       label='Protests')

# Create a plot for Riot events
df_geo[df_geo['event_type'] == 'Riots'].plot(ax=ax, 
                                       markersize=12, 
                                       color='orange', 
                                       marker='o', 
                                       label='Riots')

# Create a plot for Battle events
df_geo[df_geo['event_type'] == 'Battles'].plot(ax=ax, 
                                       markersize=12, 
                                       color='red', 
                                       marker='x', 
                                       label='Battles')

# Create a plot for Explosion events
df_geo[df_geo['event_type'] == 'Explosions/Remote violence'].plot(ax=ax, 
                                       markersize=12, 
                                       color='darkred', 
                                       marker='*', 
                                       label='Explosions')

# Create a plot for Violence against civilians events
df_geo[df_geo['event_type'] == 'Violence against civilians'].plot(ax=ax, 
                                       markersize=12, 
                                       color='darkred', 
                                       marker='^',
                                       label='Violence against civilians')

## Create a basemap of the country

# Download the shapeid geometries for the country
df_basemap = gpd.read_file(shapefile_path)

# Create the basemap plot
df_basemap["geometry"].boundary.plot(ax=ax) # If you want just an outline

## Add a scale bar (trying to fololow https://geopandas.org/en/stable/gallery/matplotlib_scalebar.html#
## and utterly failing)
#from matplotlib_scalebar.scalebar import ScaleBar
#
# Do math to figure out the distance between two points in this projection
#from shapely.geometry.point import Point
#points = gpd.GeoSeries(
#    [Point(-73.5, 40.5), Point(-74.5, 40.5)], crs=4326
#)  
#points = points.to_crs(epsg=4326)
#distance_meters = points[0].distance(points[1])
#ax.add_artist(ScaleBar(distance_meters,'km'))

# Add tile and legend
plt.title(title)
ax.legend()

# Turn off the axes, since they don't contain useful info
ax.axis('off')

# Export
#(notebook_dir.parent / "outputs" / "plots").mkdir(parents=True, exist_ok=True)
#plt.savefig(notebook_dir.parent / "outputs" / "plots" / f"{title}.png")

## Show everything
# Show all created axes simultaneously.
# If you don't do this all in the same cell, the figure resets in the next cell.
plt.show()